In [ ]:
# Global CSS for Voila dashboard
from IPython.display import display, HTML

display(HTML('''
<style>
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; }
.test-banner {
    background: #1e88e5; color: #fff; padding: 14px 18px;
    border-radius: 6px; font-size: 16px; font-weight: 600;
    margin-bottom: 12px;
}
.test-kv { font-size: 13px; line-height: 1.7; padding: 8px 14px; }
.test-kv b { display: inline-block; min-width: 140px; color: #555; }
</style>
'''))

In [ ]:
%%capture
try:
    import pandas, xnat
except ImportError:
    %pip install -q pandas xnat

In [ ]:
import pandas as pd
import xnat
import os
import requests
from IPython.display import display

# Configuration
PROJECT_ID = os.environ.get('XNAT_PROJECT', 'YOUR_PROJECT_ID')
XNAT_HOST = os.environ.get('XNAT_HOST')
XNAT_USER = os.environ.get('XNAT_USER')
XNAT_PASS = os.environ.get('XNAT_PASS')

def _apply_tls_patch():
    """Monkey-patch requests.Session for legacy SSL servers."""
    import ssl
    from requests.adapters import HTTPAdapter
    from urllib3.util.ssl_ import create_urllib3_context

    class TLSAdapter(HTTPAdapter):
        def init_poolmanager(self, *args, **kwargs):
            ctx = create_urllib3_context()
            ctx.check_hostname = False
            ctx.verify_mode = ssl.CERT_NONE
            ctx.set_ciphers('DEFAULT:@SECLEVEL=1')
            try:
                ctx.options |= ssl.OP_LEGACY_SERVER_CONNECT
            except AttributeError:
                pass
            kwargs['ssl_context'] = ctx
            return super().init_poolmanager(*args, **kwargs)

    _orig_init = requests.Session.__init__
    def _patched_init(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        self.mount('https://', TLSAdapter())
        self.verify = False
    requests.Session.__init__ = _patched_init

def _get_jsession():
    resp = requests.post(
        f'{XNAT_HOST}/data/JSESSION',
        auth=(XNAT_USER, XNAT_PASS),
        verify=False,
    )
    resp.raise_for_status()
    return resp.text

try:
    jsession_id = _get_jsession()
except Exception as e:
    if 'ssl' in type(e).__name__.lower() or 'ssl' in str(e).lower():
        _apply_tls_patch()
        jsession_id = _get_jsession()
    else:
        raise

session = xnat.connect(
    server=XNAT_HOST,
    user=XNAT_USER,
    jsession=jsession_id,
    detect_redirect=False,
    verify=False,
)
project = session.projects[PROJECT_ID]

In [ ]:
# Minimal dashboard body — confirms voila served the page and XNAT auth worked.
from IPython.display import display, HTML

subject_count = "?"
experiment_count = "?"
try:
    subject_count = len(project.subjects)
except Exception as e:
    subject_count = f"error: {e}"

try:
    experiment_count = len(project.experiments)
except Exception as e:
    experiment_count = f"error: {e}"

project_name = getattr(project, 'name', None) or PROJECT_ID

display(HTML(f"""
<div class='test-banner'>Test dashboard rendered successfully &#x2714;</div>
<div class='test-kv'>
  <div><b>Project ID</b> {PROJECT_ID}</div>
  <div><b>Project name</b> {project_name}</div>
  <div><b>XNAT host</b> {XNAT_HOST}</div>
  <div><b>XNAT user</b> {XNAT_USER}</div>
  <div><b>Subjects</b> {subject_count}</div>
  <div><b>Experiments</b> {experiment_count}</div>
</div>
"""))

In [ ]:
session.disconnect()
_ = requests.delete(f'{XNAT_HOST}/data/JSESSION', cookies={'JSESSIONID': jsession_id}, verify=False)